<a href="https://colab.research.google.com/github/Ebratul/Python/blob/main/GEN_AI/langchain/Vector_Stores_in_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
!pip install langchain langchain-community langchain-openai chromadb pypdf tiktoken openai

In [54]:
!pip install pypdf

In [70]:
!pip install pinecone langchain-pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.1
    Uninstalling packaging-26.1:
      Successfully uninstalled packaging-26.1
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.41.1 which is i

In [76]:
!pip install pinecone langchain-pinecone sentence-transformers

In [71]:
# from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_community.vectorstores import Chroma

from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

In [72]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [73]:
doc = [doc1, doc2, doc3, doc4, doc5]

In [74]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [59]:
vector_store = Chroma(
    embedding_function=embedding,
    persist_directory="chroma_db",
    collection_name="sample"
)

In [77]:
from langchain_pinecone import  PineconeVectorStore

In [79]:
import os
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings

os.environ["PINECONE_API_KEY"] = "your_api_key"

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = "sample"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = PineconeVectorStore(
    index_name=index_name,
    embedding=embedding
)

ValueError: Pinecone API key must be provided in either `pinecone_api_key` or `PINECONE_API_KEY` environment variable

In [60]:
# add document
vector_store.add_documents(doc)

['1fbb366f-f580-4df7-9cf5-90b3f9f667ba',
 '5408ef8d-ced0-4629-93b0-ee395ac87efd',
 '8c4abca2-9696-41a8-98c2-96975468c693',
 '29a51033-ed26-4c94-bd37-5d112afccb04',
 '6cc7880e-c6df-4bc5-8875-d03f482ee22c']

In [61]:
# view document
df = vector_store.get(include = ['embeddings', 'documents', 'metadatas'])

In [62]:
df['embeddings'][0]

array([ 1.27745932e-03,  3.12985331e-02, -2.37537846e-02,  1.18730534e-02,
       -1.11592319e-02,  5.63993417e-02,  5.62181100e-02,  2.74404045e-02,
        3.20060588e-02,  3.70754153e-02, -9.66248512e-02, -4.98729274e-02,
        5.42407446e-02,  5.93064614e-02,  4.75408472e-02, -9.67528392e-03,
        2.06799470e-02, -4.59902957e-02,  1.45080937e-02, -1.14381202e-01,
       -4.47647497e-02,  6.55564889e-02, -3.95612605e-02, -6.55716732e-02,
        1.31235979e-02, -4.36901674e-02,  1.23226214e-02, -3.78008885e-03,
       -2.25683041e-02, -1.18990012e-01,  5.17058969e-02,  1.17330458e-02,
        5.30784205e-02, -2.27654376e-03, -8.26016068e-02,  8.16718582e-03,
       -1.77791389e-03,  7.05591217e-02,  5.32849878e-02, -2.54713334e-02,
        3.99076715e-02,  4.60824789e-03,  1.71810221e-02, -6.47474825e-02,
       -3.04842852e-02, -7.66589344e-02, -5.38962567e-03, -1.84742045e-02,
        2.74487566e-02, -3.07490081e-02, -9.29607749e-02,  2.48330813e-02,
        5.72462194e-02, -

In [63]:
# search document

vector_store.similarity_search(
    query = 'Who among there are a bowler?',
    k = 2
)

[Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [64]:
# search with similarity score

vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k = 2
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693599343299866),
 (Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693599343299866)]

In [65]:
# meta-data filtering

vector_store.similarity_search_with_score(
    query = "",
    filter = {"team" : "Chennai Super Kings"}
)

[(Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436005115509033),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436005115509033),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.890937328338623),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-w

In [66]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='55575859-553d-413c-80f4-36bd25f40094', document=updated_doc1)

In [67]:
# view document

vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['1b68848f-938e-411b-a489-791e1bb796fd',
  '8702436b-6ebf-452d-9326-c1acca54a521',
  '61592937-d24e-47b7-9188-d04fd568829c',
  '6a97a49b-b2f2-4cd6-bd62-ee15f260150e',
  '1fbb366f-f580-4df7-9cf5-90b3f9f667ba',
  '5408ef8d-ced0-4629-93b0-ee395ac87efd',
  '8c4abca2-9696-41a8-98c2-96975468c693',
  '29a51033-ed26-4c94-bd37-5d112afccb04',
  '6cc7880e-c6df-4bc5-8875-d03f482ee22c'],
 'embeddings': array([[ 0.00127746,  0.03129853, -0.02375378, ..., -0.0051836 ,
         -0.03280611,  0.02737715],
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123395, -0.02468549, -0.04494376, ..., -0.10995813,
          0.00572561,  0.09915381],
        ...,
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123395, -0.02468549, -0.04494376, ..., -0.10995813,
          0.00572561,  0.09915381],
        [ 0.0187398 ,  0.04382842, -0.04304253, ..., -0.0780162 ,
         -0

In [68]:
vector_store.delete(ids = ['55575859-553d-413c-80f4-36bd25f40094'])

In [69]:
from chromadb import Include
# view document

vector_store.get(include = ['embeddings', 'documents', 'metadatas'])

{'ids': ['1b68848f-938e-411b-a489-791e1bb796fd',
  '8702436b-6ebf-452d-9326-c1acca54a521',
  '61592937-d24e-47b7-9188-d04fd568829c',
  '6a97a49b-b2f2-4cd6-bd62-ee15f260150e',
  '1fbb366f-f580-4df7-9cf5-90b3f9f667ba',
  '5408ef8d-ced0-4629-93b0-ee395ac87efd',
  '8c4abca2-9696-41a8-98c2-96975468c693',
  '29a51033-ed26-4c94-bd37-5d112afccb04',
  '6cc7880e-c6df-4bc5-8875-d03f482ee22c'],
 'embeddings': array([[ 0.00127746,  0.03129853, -0.02375378, ..., -0.0051836 ,
         -0.03280611,  0.02737715],
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123395, -0.02468549, -0.04494376, ..., -0.10995813,
          0.00572561,  0.09915381],
        ...,
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123395, -0.02468549, -0.04494376, ..., -0.10995813,
          0.00572561,  0.09915381],
        [ 0.0187398 ,  0.04382842, -0.04304253, ..., -0.0780162 ,
         -0